# TAMPO Algorithms — Colab Test Run Notebook

This notebook tests the currently implemented TAMPO meta-RL algorithms (TAMPO-GCN and TAMPO-LSTM).
Run cells **top to bottom** in order.

## 0. Setup — Clone repo & install dependencies

In [ ]:
# ── 0a. Clone the repo ────────────────────────────────────────────
!git clone -b gcn-pyg https://github.com/vikkesh/tampo.git tampo
%cd tampo

In [ ]:
!nvidia-smi

In [ ]:
# ── 0b. Install all dependencies ─────────────────────────────────
!pip install -r requirements.txt

In [ ]:
# ── 0c. Verify imports ────────────────────────────────────────────
import torch, gymnasium as gym, yaml, json, numpy as np
from torch_geometric.data import Batch, Data
print("torch:", torch.__version__)
print("gymnasium:", gym.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Unit Test — DAGEncoder with Bi-GCN path

Verifies that the encoder:
- Accepts a PyG Batch of **variable-size** graphs
- Produces a context vector of the correct shape 
- Properly handles the bi-directional pass without crashing

In [ ]:
import sys, os
sys.path.insert(0, '.')  # tampo root

import torch
from torch_geometric.data import Batch, Data
from algorithms.rl.tampo import DAGEncoder

TASK_FEATURE_DIM = 6
HIDDEN_DIM = 128
SERVER_FEATURE_DIM = 20

encoder = DAGEncoder(
    task_feature_dim=TASK_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=2,
    encoder_type='gcn',
    server_feature_dim=SERVER_FEATURE_DIM
)
encoder.eval()

# Three graphs with DIFFERENT node counts (8, 12, 20)
graphs = []
for n in [8, 12, 20]:
    x = torch.rand(n, TASK_FEATURE_DIM)
    # Random directed edges (roughly a chain)
    src = torch.arange(0, n - 1)
    dst = torch.arange(1, n)
    edge_index = torch.stack([src, dst], dim=0)
    graphs.append(Data(x=x, edge_index=edge_index, num_nodes=n))

batch = Batch.from_data_list(graphs)

# Dummy task_features tensor
task_features_dummy = torch.zeros(3, 20, TASK_FEATURE_DIM)
server_features = torch.rand(3, SERVER_FEATURE_DIM)

with torch.no_grad():
    encoded_tasks, context = encoder(
        task_features=task_features_dummy,
        graph_batch=batch,
        server_features=server_features
    )

assert context.shape == (3, HIDDEN_DIM * 2), f"Bad context shape: {context.shape}"
print(f"PASS — context shape: {context.shape}  (expected [3, {HIDDEN_DIM*2}])")
print(f"PASS — encoded_tasks shape: {encoded_tasks.shape}")


## 2. Generate the Golden Test Dataset

> ⚠️ Run **once only**.  Never re-run between algorithm comparisons.

In [ ]:
!python utils/generate_test_dataset.py --num_dags 20 --output data/test_dags.json

import json
with open("data/test_dags.json") as f:
    ds = json.load(f)
print(f"Dataset contains {len(ds)} entries")
print("First entry keys:", list(ds[0].keys()))

## 2.5 Verification — Physics and Reward Overhaul

Confirms that the environment correctly implements dynamic server loads (Makespan) and diverse, context-sensitive rewards.

In [ ]:
import sys, yaml, numpy as np
sys.path.insert(0, '.')
from env.base_offloading_env import TaskOffloadingEnv
from utils.dag_parser import DAGParser

with open('configs/default_config.yaml') as f:
    fc = yaml.safe_load(f)
cfg = {}
env_cfg = full_config.get('environment', full_config)
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(fc['environment'][sec] if 'environment' in fc else fc[sec])

env = TaskOffloadingEnv(cfg)
dags = DAGParser('data/meta_offloading_n/offload_random20').load_dataset(num_graphs=1)
env.reset(task_graph=dags[0])

print("--- Cell A: Server Load Dynamic Check ---")
snapshots = [env.server_available.copy()]
for i in range(dags[0]['num_tasks']):
    action = i % env.action_space.n
    obs, reward, done, info = env.step(action)
    snapshots.append(env.server_available.copy())
    print(f'Node {i:2d} → server={action} | available={env.server_available.round(3)} | reward={reward:.3f}')
    if done:
        break

all_same = all(np.allclose(snapshots[0], s) for s in snapshots)
print('\n❌ FAIL — server state static.' if all_same else '\n✅ PASS — server state changes dynamically.')
print(f'Makespan: {env.total_delay:.4f}s  Energy: {env.total_energy:.6f}J')

print("\n--- Cell B: Reward Diversity Check ---")
dags_sample = DAGParser('data/meta_offloading_n/offload_random20').load_dataset(num_graphs=5)
rewards_by_action = {a: [] for a in range(env.action_space.n)}
for dag in dags_sample:
    env.reset(task_graph=dag)
    for _ in range(dag['num_tasks']):
        for a in range(env.action_space.n):
            d, e = env._execute_offloading(a)
            rewards_by_action[a].append(env._calculate_reward(d, e))
            # Undo state mutation
            env.node_finish_times[env.topo_order[env.current_node_idx]] = 0
            env.node_assignments[env.topo_order[env.current_node_idx]] = -1
for a, rs in rewards_by_action.items():
    print(f'Action {a}: mean={np.mean(rs):.3f}  min={np.min(rs):.3f}  max={np.max(rs):.3f}')


## 3. Quick Training Smoke Test — TAMPO-GCN and TAMPO-LSTM (1 iteration)

Checks that the full training loop executes without shape errors for both algorithms.

In [ ]:
import yaml, sys
sys.path.insert(0, '.')

with open('configs/default_config.yaml') as f:
    full_config = yaml.safe_load(f)

from env.base_offloading_env import TaskOffloadingEnv
from algorithms.rl.tampo import TAMPOFramework

cfg = {}
env_cfg = full_config.get('environment', full_config)
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(env_cfg.get(sec, {}))
cfg['reward'] = env_cfg.get('reward', {})

env = TaskOffloadingEnv(cfg)

# Load training graphs from raw data folders (NEVER train on test_dags.json!)
from utils.dag_parser import DAGParser
dag_parser = DAGParser(data_folder='data/meta_offloading_20/offload_random20_1')
task_graphs = dag_parser.load_dataset(num_graphs=50)

tasks_for_env = []
for dag in task_graphs:
    tasks_for_env.append({
        'num_tasks': dag['num_tasks'],
        'tasks': dag['tasks'],
        'edges': dag['edges'],
        'adj_matrix': dag['adj_matrix'],
        'size': sum((t['data_size'] for t in dag['tasks']), start=0),
        'cycles': sum((t['cycles'] for t in dag['tasks']), start=0)
    })
env.load_task_dataset(tasks_for_env)

for enc_type in ['gcn', 'lstm']:
    print(f'\n======================================')
    print(f'Testing TAMPO-{enc_type.upper()}')
    print(f'======================================')
    tampo_cfg = full_config['algorithms']['tampo'].copy()
    tampo_cfg['encoder_type'] = enc_type
    tampo_cfg['num_meta_iterations'] = 1  # just 1 iteration for smoke test

    framework = TAMPOFramework(env, tampo_cfg)
    results = framework.train(num_iterations=1, meta_batch_size=2)
    
    import os
    os.makedirs('models', exist_ok=True)
    framework.save(f'models/tampo_{enc_type}_checkpoint.pth')
    print(f'SMOKE TEST PASSED for TAMPO-{enc_type.upper()} — checkpoint saved.')


## 4. Run Benchmark on Trained Models

Compares both trained algorithms against the same test dataset.

In [ ]:
import os, sys
sys.path.insert(0, '.')

# Run benchmark for both algorithms
!python benchmark.py \
    --algorithms TAMPO_GCN TAMPO_LSTM \
    --checkpoint_dir models/ \
    --dataset_path data/test_dags.json \
    --output_dir results/


## 5. Download Results

In [ ]:
import os
import glob
from google.colab import files

# Find the most recently created run directory
run_dirs = sorted(glob.glob('results/run_*'), reverse=True)
if run_dirs:
    latest_run = run_dirs[0]
    for fname in ['comparison_bar.png', 'pareto_front.png', 'benchmark_results.csv']:
        path = os.path.join(latest_run, fname)
        if os.path.exists(path):
            files.download(path)
            print(f'Downloaded {path}')
        else:
            print(f'Not found: {path}')
else:
    print('No results found. Run benchmark first.')
